# ETLegacy Core Infrastructure

Generate a small Einstein Toolkit-style thorn and inspect its CCL and source
files.

Navigation:
[Index](../index.ipynb) |
Previous: [BHaH Project Anatomy](bhah_project_anatomy.ipynb) |
Next: [Carpet WaveToy Thorns](carpet_wavetoy_thorns.ipynb)

## Learning Goals

- Create a small Einstein Toolkit-style thorn from NRPy ingredients.
- Read CCL files as declarations of variables, parameters, schedules, and source.
- Connect scheduled functions to the time-step workflow.

## Words for This Notebook

- **Thorn:** an Einstein Toolkit component folder.
- **CCL file:** a text file used by the Einstein Toolkit to describe a thorn.
- **Schedule:** the order and time when functions run.
- **Hook:** a function placed into a larger workflow at a named point.
- **Registry:** a stored list of named objects that NRPy later writes to files.
- **Installed package:** a Python package imported from `site-packages`.

Use the code cells actively: first predict what should happen, then run the cell.
Afterward, explain the output in plain language.

## Table of Contents

- [Generated Thorn File Roles](#Generated-Thorn-File-Roles)
- [Step 1: Import File and Output Helpers](#Step-1:-Import-File-and-Output-Helpers)
- [Step 2: Create a Thorn Workspace](#Step-2:-Create-a-Thorn-Workspace)
- [Step 3: Import ETLegacy Registration Tools](#Step-3:-Import-ETLegacy-Registration-Tools)
- [Step 4: Register Grid Functions](#Step-4:-Register-Grid-Functions)
- [Step 5: Build the Right-Hand-Side Body](#Step-5:-Build-the-Right-Hand-Side-Body)
- [Step 6: Register the RHS C Function](#Step-6:-Register-the-RHS-C-Function)
- [Step 7: Register Workflow Hooks](#Step-7:-Register-Workflow-Hooks)
- [Step 8: Write ETLegacy Thorn Files](#Step-8:-Write-ETLegacy-Thorn-Files)
- [Step 9: Read Generator Messages](#Step-9:-Read-Generator-Messages)
- [Validation Check](#Validation-Check)
- [Step 10: Inspect the Generated Inventory](#Step-10:-Inspect-the-Generated-Inventory)
- [Step 11: Read `interface.ccl`](#Step-11:-Read-interface.ccl)
- [Step 12: Read `param.ccl`](#Step-12:-Read-param.ccl)
- [Step 13: Read `schedule.ccl`](#Step-13:-Read-schedule.ccl)
- [Step 14: Read `make.code.defn`](#Step-14:-Read-make.code.defn)
- [Step 15: Read the RHS Source](#Step-15:-Read-the-RHS-Source)

## Generated Thorn File Roles

| File | Role | What to inspect |
| --- | --- | --- |
| `interface.ccl` | Variables and inherited capabilities | Grid field groups |
| `param.ccl` | Runtime parameters | `wave_speed` declaration |
| `schedule.ccl` | Workflow order | Schedule bins and hooks |
| `src/make.code.defn` | Build metadata | Source files compiled |
| `src/*_rhs_eval.c` | Generated C function | Parameter read and grid loop |

## Step 1: Import File and Output Helpers

These standard-library tools capture generator messages, manage temporary project
directories, and clean command output.

In [1]:
from pathlib import Path
import contextlib
import io
import os
import re
import tempfile
def local_nrpy_source_roots():
    roots = []
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / "nrpy"
        if (candidate / "nrpy" / "__init__.py").is_file():
            roots.append(candidate.resolve())
    return roots


def require_installed_nrpy(import_path):
    resolved = Path(import_path).resolve()
    for source_root in local_nrpy_source_roots():
        if resolved.is_relative_to(source_root):
            raise RuntimeError(f"NRPy loaded from local source tree: {resolved}")
    installed_markers = {"site-packages", "dist-packages"}
    if not installed_markers.intersection(resolved.parts):
        raise RuntimeError(f"NRPy did not load from an installed package: {resolved}")
    return resolved

## Step 2: Create a Thorn Workspace

The workspace keeps generated files and the NRPy cache separate from the tutorial
source tree.

In [2]:
workspace_manager = tempfile.TemporaryDirectory(
    prefix="nrpy_tutorial_etlegacy_", dir=Path.cwd()
)
WORKSPACE = Path(workspace_manager.name)
CACHE_DIR = WORKSPACE / ".cache"
CACHE_DIR.mkdir()
os.environ["XDG_CACHE_HOME"] = str(CACHE_DIR)
PROJECT_DIR = WORKSPACE / "project" / "toy_etlegacy"
THORN_NAME = "ToyETLegacyNRPy"

print("workspace:", WORKSPACE)
print("project path:", PROJECT_DIR)
print("cache:", CACHE_DIR)

workspace: /work/5-infrastructures/nrpy_tutorial_etlegacy_f_asbxw3
project path: /work/5-infrastructures/nrpy_tutorial_etlegacy_f_asbxw3/project/toy_etlegacy
cache: /work/5-infrastructures/nrpy_tutorial_etlegacy_f_asbxw3/.cache


## Step 3: Import ETLegacy Registration Tools

These imports expose the NRPy registries and infrastructure writers used below.

In [3]:
import nrpy
import nrpy.c_function as cfc
import nrpy.grid as gri
import nrpy.params as par
from nrpy.infrastructures.ETLegacy import (
    CodeParameters,
    MoL_registration,
    Symmetry_registration,
    boundary_conditions,
    interface_ccl,
    make_code_defn,
    param_ccl,
    schedule_ccl,
    simple_loop,
    zero_rhss,
)

nrpy_import_path = require_installed_nrpy(nrpy.__file__)
print("NRPy installed package:", nrpy_import_path)
print("ETLegacy tools imported for thorn:", THORN_NAME)

NRPy installed package: /virt/lib/python3.12/site-packages/nrpy/__init__.py
ETLegacy tools imported for thorn: ToyETLegacyNRPy


## Step 4: Register Grid Functions

The registry records named grid fields and their roles in generated code.

In [4]:
for name in [
    f"{THORN_NAME}_rhs_eval",
    f"{THORN_NAME}_zero_rhss",
    f"{THORN_NAME}_MoL_registration",
    f"{THORN_NAME}_Symmetry_registration_oldCartGrid3D",
    f"{THORN_NAME}_specify_Driver_BoundaryConditions",
]:
    cfc.CFunction_dict.pop(name, None)
for name in ["psi", "energy", "psi_rhs", "energy_rhs"]:
    gri.glb_gridfcs_dict.pop(name, None)
par.set_parval_from_str("Infrastructure", "ETLegacy")
par.register_CodeParameter("CCTK_INT", __name__, "fd_order", 4)
par.register_CodeParameter("CCTK_REAL", __name__, "wave_speed", 1.0)
evolved_fields = gri.register_gridfunctions(["psi"], group="EVOL")
aux_fields = gri.register_gridfunctions(["energy"], group="AUX")

print("evolved grid fields:", [str(field) for field in evolved_fields])
print("auxiliary grid fields:", [str(field) for field in aux_fields])

evolved grid fields: ['psi']
auxiliary grid fields: ['energy']


## Step 5: Build the Right-Hand-Side Body

This body combines ETLegacy parameter reads with a loop over grid points.

In [5]:
body = f"""DECLARE_CCTK_ARGUMENTS_{THORN_NAME}_rhs_eval;
DECLARE_CCTK_PARAMETERS;
"""
body += CodeParameters.read_CodeParameters(
    list_of_tuples__thorn_CodeParameter=[(THORN_NAME, "wave_speed")],
    declare_invdxxs=True,
)
body += simple_loop.simple_loop(
    loop_body=(
        f"{gri.ETLegacyGridFunction.access_gf('psi_rhs')} = "
        f"wave_speed * ({gri.ETLegacyGridFunction.access_gf('energy')} - "
        f"{gri.ETLegacyGridFunction.access_gf('psi')});"
    ),
    loop_region="interior",
)

print("right-hand-side body:")
print(body)

right-hand-side body:
DECLARE_CCTK_ARGUMENTS_ToyETLegacyNRPy_rhs_eval;
DECLARE_CCTK_PARAMETERS;
const CCTK_REAL *wave_speedptr = CCTK_ParameterGet("wave_speed", "ToyETLegacyNRPy", NULL);  // ToyETLegacyNRPy::wave_speed
const CCTK_REAL wave_speed = *wave_speedptr;
const CCTK_REAL invdxx0 = 1.0/CCTK_DELTA_SPACE(0);
const CCTK_REAL invdxx1 = 1.0/CCTK_DELTA_SPACE(1);
const CCTK_REAL invdxx2 = 1.0/CCTK_DELTA_SPACE(2);
#pragma omp parallel for
for (int i2 = cctk_nghostzones[2]; i2 < cctk_lsh[2]-cctk_nghostzones[2]; i2++) {
for (int i1 = cctk_nghostzones[1]; i1 < cctk_lsh[1]-cctk_nghostzones[1]; i1++) {
for (int i0 = cctk_nghostzones[0]; i0 < cctk_lsh[0]-cctk_nghostzones[0]; i0++) {
psi_rhsGF[CCTK_GFINDEX3D(cctkGH, i0, i1, i2)] = wave_speed * (energyGF[CCTK_GFINDEX3D(cctkGH, i0, i1, i2)] - psiGF[CCTK_GFINDEX3D(cctkGH, i0, i1, i2)]);
} // END LOOP: for i0 over [cctk_nghostzones[0], cctk_lsh[0]-cctk_nghostzones[0])
} // END LOOP: for i1 over [cctk_nghostzones[1], cctk_lsh[1]-cctk_nghostzones[1]

## Step 6: Register the RHS C Function

The registration connects the generated C function to the `MoL_CalcRHS` schedule
bin.

In [6]:
cfc.register_CFunction(
    subdirectory=THORN_NAME,
    includes=["cctk.h", "cctk_Arguments.h", "cctk_Parameters.h"],
    desc="Evaluate a toy right-hand side.",
    cfunc_type="void",
    name=f"{THORN_NAME}_rhs_eval",
    params="CCTK_ARGUMENTS",
    body=body,
    ET_thorn_name=THORN_NAME,
    ET_schedule_bins_entries=[
        (
            "MoL_CalcRHS",
            """
schedule FUNC_NAME in MoL_CalcRHS
{
  LANG: C
  READS: evol_variables(everywhere)
  READS: aux_variables(everywhere)
  WRITES: evol_variables_rhs(interior)
} "Evaluate toy right-hand side"
""",
        )
    ],
    ET_current_thorn_CodeParams_used=["wave_speed"],
)
print(cfc.CFunction_dict[f"{THORN_NAME}_rhs_eval"].function_prototype)

void ToyETLegacyNRPy_rhs_eval(CCTK_ARGUMENTS);


## Step 7: Register Workflow Hooks

These helper registrations add zero-RHS, Method of Lines, symmetry, and boundary
functions to the thorn.

In [7]:
zero_rhss.register_CFunction_zero_rhss(THORN_NAME)
MoL_registration.register_CFunction_MoL_registration(THORN_NAME)
Symmetry_registration.register_CFunction_Symmetry_registration_oldCartGrid3D(THORN_NAME)
boundary_conditions.register_CFunction_specify_Driver_BoundaryConditions(
    thorn_name=THORN_NAME
)
print("registered C functions:")
for name in sorted(cfc.CFunction_dict):
    print(name)

registered C functions:
ToyETLegacyNRPy_MoL_registration
ToyETLegacyNRPy_Symmetry_registration_oldCartGrid3D
ToyETLegacyNRPy_rhs_eval
ToyETLegacyNRPy_specify_Driver_BoundaryConditions
ToyETLegacyNRPy_zero_rhss


## Step 8: Write ETLegacy Thorn Files

The constructors write the CCL declarations and generated source files for the
thorn.

In [8]:
generation_output = io.StringIO()
uses_includes = (
    "USES INCLUDE: Symmetry.h" + chr(10)
    + "USES INCLUDE: Boundary.h" + chr(10)
)
with contextlib.redirect_stdout(generation_output):
    interface_ccl.construct_interface_ccl(
        project_dir=str(PROJECT_DIR),
        thorn_name=THORN_NAME,
        inherits="Boundary Grid MethodofLines",
        USES_INCLUDEs=uses_includes,
        is_evol_thorn=True,
    )
    param_ccl.construct_param_ccl(project_dir=str(PROJECT_DIR), thorn_name=THORN_NAME)
    schedule_ccl.construct_schedule_ccl(
        project_dir=str(PROJECT_DIR),
        thorn_name=THORN_NAME,
        STORAGE="""
STORAGE: evol_variables[3]
STORAGE: evol_variables_rhs[1]
STORAGE: aux_variables[1]
""",
    )
    make_code_defn.output_CFunctions_and_construct_make_code_defn(
        project_dir=str(PROJECT_DIR), thorn_name=THORN_NAME
    )

print("wrote thorn files to:", PROJECT_DIR / THORN_NAME)
print("captured generator messages:", len(generation_output.getvalue().splitlines()))

wrote thorn files to: /work/5-infrastructures/nrpy_tutorial_etlegacy_f_asbxw3/project/toy_etlegacy/ToyETLegacyNRPy
captured generator messages: 9


## Step 9: Read Generator Messages

The cleaned messages confirm which files the infrastructure writer produced.

In [9]:
cleaned = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", generation_output.getvalue())
cleaned = cleaned.replace(str(WORKSPACE), "<workspace>").replace(
    str(PROJECT_DIR), "project/toy_etlegacy"
)
print("generation output:")
for line in cleaned.splitlines():
    if line.strip():
        print(line.rstrip())

generation output:
Checking <workspace>/project/toy_etlegacy/ToyETLegacyNRPy/interface.ccl...[written]
Checking <workspace>/project/toy_etlegacy/ToyETLegacyNRPy/param.ccl...[written]
Checking <workspace>/project/toy_etlegacy/ToyETLegacyNRPy/schedule.ccl...[written]
Checking <workspace>/project/toy_etlegacy/ToyETLegacyNRPy/src/ToyETLegacyNRPy_MoL_registration.c...[written]
Checking <workspace>/project/toy_etlegacy/ToyETLegacyNRPy/src/ToyETLegacyNRPy_Symmetry_registration_oldCartGrid3D.c...[written]
Checking <workspace>/project/toy_etlegacy/ToyETLegacyNRPy/src/ToyETLegacyNRPy_rhs_eval.c...[written]
Checking <workspace>/project/toy_etlegacy/ToyETLegacyNRPy/src/ToyETLegacyNRPy_specify_Driver_BoundaryConditions.c...[written]
Checking <workspace>/project/toy_etlegacy/ToyETLegacyNRPy/src/ToyETLegacyNRPy_zero_rhss.c...[written]
Checking <workspace>/project/toy_etlegacy/ToyETLegacyNRPy/src/make.code.defn...[written]


## Validation Check

The guard stops the notebook with a clear missing-file error if generation did
not produce the expected artifacts.

In [10]:
required_files = [
    "interface.ccl",
    "param.ccl",
    "schedule.ccl",
    "src/make.code.defn",
    f"src/{THORN_NAME}_rhs_eval.c",
]
missing = [
    relative_path
    for relative_path in required_files
    if not (PROJECT_DIR / THORN_NAME / relative_path).is_file()
]
if missing:
    raise FileNotFoundError("Missing generated files: " + ", ".join(missing))
print("validation passed: required ETLegacy thorn files are present")

validation passed: required ETLegacy thorn files are present


## Step 10: Inspect the Generated Inventory

The inventory identifies the generated files relevant to this lesson.

In [11]:
print("thorn inventory:")
for path in sorted((PROJECT_DIR / THORN_NAME).rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT_DIR / THORN_NAME))

thorn inventory:
interface.ccl
param.ccl
schedule.ccl
src/ToyETLegacyNRPy_MoL_registration.c
src/ToyETLegacyNRPy_Symmetry_registration_oldCartGrid3D.c
src/ToyETLegacyNRPy_rhs_eval.c
src/ToyETLegacyNRPy_specify_Driver_BoundaryConditions.c
src/ToyETLegacyNRPy_zero_rhss.c
src/make.code.defn


## Step 11: Read `interface.ccl`

The `interface.ccl` file declares grid variables and inherited capabilities.

In [12]:
print(
    (PROJECT_DIR / THORN_NAME / "interface.ccl").read_text(
        encoding="utf-8", errors="replace"
    )
)


# This interface.ccl file was automatically generated by NRPy.
#   You are advised against modifying it directly; instead
#   modify the Python code that generates it.

# With "implements", we give our thorn its unique name.
implements: ToyETLegacyNRPy

# By "inheriting" other thorns, we tell the Toolkit that we
#   will rely on variables/function that exist within those
#   functions.
inherits: Boundary Grid MethodofLines

# Needed functions and #include's:
USES INCLUDE: Symmetry.h
USES INCLUDE: Boundary.h


# Needed Method of Lines function
CCTK_INT FUNCTION MoLRegisterEvolvedGroup(CCTK_INT IN EvolvedIndex, CCTK_INT IN RHSIndex)
REQUIRES FUNCTION MoLRegisterEvolvedGroup

# Needed Boundary Conditions function
CCTK_INT FUNCTION GetBoundarySpecification(CCTK_INT IN size, CCTK_INT OUT ARRAY nboundaryzones, CCTK_INT OUT ARRAY is_internal, CCTK_INT OUT ARRAY is_staggered, CCTK_INT OUT ARRAY shiftout)
USES FUNCTION GetBoundarySpecification

CCTK_INT FUNCTION SymmetryTableHandleForGrid(CCTK

## Step 12: Read `param.ccl`

The parameter file declares runtime parameters exposed by the thorn.

In [13]:
print(
    (PROJECT_DIR / THORN_NAME / "param.ccl").read_text(
        encoding="utf-8", errors="replace"
    )
)

# This param.ccl file was automatically generated by NRPy.
#   You are advised against modifying it directly; instead
#   modify the Python code that generates it.


restricted:
CCTK_INT fd_order "(see NRPy for parameter definition)"
{
 *:* :: "All values accepted. NRPy does not restrict the allowed ranges of parameters yet."
} 4

CCTK_REAL wave_speed "(see NRPy for parameter definition)"
{
 *:* :: "All values accepted. NRPy does not restrict the allowed ranges of parameters yet."
} 1.0




## Step 13: Read `schedule.ccl`

The schedule file places functions in Einstein Toolkit schedule bins.

In [14]:
print(
    (PROJECT_DIR / THORN_NAME / "schedule.ccl").read_text(
        encoding="utf-8", errors="replace"
    )
)

# This schedule.ccl file was automatically generated by NRPy.
#   You are advised against modifying it directly; instead
#   modify the Python code that generates it.

##################################################
# Step 0: Allocate memory for gridfunctions, using the STORAGE: keyword.

STORAGE: evol_variables[3]
STORAGE: evol_variables_rhs[1]
STORAGE: aux_variables[1]


##################################################
# Step 1: Schedule functions in the Driver_BoundarySelect scheduling bin.

schedule ToyETLegacyNRPy_specify_Driver_BoundaryConditions in Driver_BoundarySelect
{
  LANG: C
  OPTIONS: meta
} "Register boundary conditions in PreSync bin Driver_BoundarySelect."

##################################################
# Step 2: Schedule functions in the BASEGRID scheduling bin.

schedule ToyETLegacyNRPy_Symmetry_registration_oldCartGrid3D at BASEGRID as Symmetry_registration
{
  LANG: C
  OPTIONS: Global
} "Register symmetries, the CartGrid3D way."

schedule ToyETLegacyNRPy

## Step 14: Read `make.code.defn`

This file lists source files compiled into the thorn.

In [15]:
print(
    (PROJECT_DIR / THORN_NAME / "src" / "make.code.defn").read_text(
        encoding="utf-8", errors="replace"
    )
)

# make.code.defn file for thorn ToyETLegacyNRPy

# Source files that need to be compiled:
SRCS = \
       ToyETLegacyNRPy_MoL_registration.c \
       ToyETLegacyNRPy_Symmetry_registration_oldCartGrid3D.c \
       ToyETLegacyNRPy_rhs_eval.c \
       ToyETLegacyNRPy_specify_Driver_BoundaryConditions.c \
       ToyETLegacyNRPy_zero_rhss.c



## Step 15: Read the RHS Source

The full source file is the generated C function registered earlier. Look for the
includes, scheduled function name, parameter read, grid loop, and toy assignment.

In [16]:
print(
    (PROJECT_DIR / THORN_NAME / "src" / f"{THORN_NAME}_rhs_eval.c").read_text(
        encoding="utf-8", errors="replace"
    )
)

#include "cctk.h"
#include "cctk_Arguments.h"
#include "cctk_Parameters.h"

/**
 * Evaluate a toy right-hand side.
 */
void ToyETLegacyNRPy_rhs_eval(CCTK_ARGUMENTS) {
  DECLARE_CCTK_ARGUMENTS_ToyETLegacyNRPy_rhs_eval;
  DECLARE_CCTK_PARAMETERS;
  const CCTK_REAL *wave_speedptr = CCTK_ParameterGet("wave_speed", "ToyETLegacyNRPy", NULL); // ToyETLegacyNRPy::wave_speed
  const CCTK_REAL wave_speed = *wave_speedptr;
  const CCTK_REAL invdxx0 = 1.0 / CCTK_DELTA_SPACE(0);
  const CCTK_REAL invdxx1 = 1.0 / CCTK_DELTA_SPACE(1);
  const CCTK_REAL invdxx2 = 1.0 / CCTK_DELTA_SPACE(2);
#pragma omp parallel for
  for (int i2 = cctk_nghostzones[2]; i2 < cctk_lsh[2] - cctk_nghostzones[2]; i2++) {
    for (int i1 = cctk_nghostzones[1]; i1 < cctk_lsh[1] - cctk_nghostzones[1]; i1++) {
      for (int i0 = cctk_nghostzones[0]; i0 < cctk_lsh[0] - cctk_nghostzones[0]; i0++) {
        psi_rhsGF[CCTK_GFINDEX3D(cctkGH, i0, i1, i2)] =
            wave_speed * (energyGF[CCTK_GFINDEX3D(cctkGH, i0, i1, i2)] - psiG

The printed CCL and source files show how the same registered C-function idea is
expressed as Einstein Toolkit thorn metadata. The schedule entries state when
each generated function runs.

## Learning Check

Before reading the CCL files, predict which file should describe variables,
which should describe parameters, and which should describe when functions run.

## Continue to Carpet WaveToy Thorns

- [C Function Registration](../1-intro/c_function.ipynb)
- [BHaH Project Anatomy](bhah_project_anatomy.ipynb)
- [Carpet WaveToy Thorns](carpet_wavetoy_thorns.ipynb)
- [Code-Writing Path Choice Guide](backend_choice_guide.ipynb)